In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="08-ad-click-prediction",
    notebook_name="02_feature_engineering_deep_dive.ipynb"
)

# 02 -- Feature Engineering Deep Dive for Ad Click Prediction

**Goal:** Master the feature engineering techniques that make or break an ad click prediction system.

---

## The Toy Store Analogy

Imagine you run a toy store and you want to predict which kid will buy which toy.

- **Sparse features:** Each kid can be described by thousands of attributes (favorite color, school, birthday month, favorite cartoon character...) but most of those are "not applicable" for any given kid. That is sparsity.
- **Dense features:** Instead of listing every possible cartoon character, you could describe a kid's taste as a point in "cartoon space" -- like coordinates on a map. That is a dense embedding.
- **Feature crosses:** A 5-year-old boy who likes dinosaurs is VERY different from a 5-year-old girl who likes dinosaurs in terms of what toy they will pick. The combination matters more than each feature alone.

This notebook dives deep into all of these techniques for ad click prediction.

## 1. Sparse vs Dense Features

### What Makes Ad Systems Special

Most ML problems have nice, tidy numerical features. Ad click prediction has MILLIONS of categorical features, and most of them are zero.

| Feature Type | Example | Cardinality | Representation |
|-------------|---------|-------------|----------------|
| User ID | `user_38291047` | ~1 billion | Sparse (one-hot) |
| Ad ID | `ad_502938` | ~10 million | Sparse (one-hot) |
| Ad Category | `travel` | ~100 | Sparse (one-hot) |
| Hour of Day | `14` | 24 | Dense (numerical) |
| Historical CTR | `0.032` | Continuous | Dense (numerical) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix

# === Visualizing Sparsity ===
# Let's see what a one-hot encoded feature vector looks like
# for a typical ad impression

# Simulate a feature space for one impression
np.random.seed(42)

# Cardinalities of different feature groups
feature_groups = {
    'user_id': 1_000_000,       # 1M unique users
    'ad_id': 500_000,           # 500K unique ads
    'advertiser_id': 50_000,    # 50K advertisers
    'campaign_id': 200_000,     # 200K campaigns
    'ad_category': 100,         # 100 categories
    'ad_subcategory': 1_000,    # 1000 subcategories
    'user_city': 50_000,        # 50K cities
    'user_country': 200,        # 200 countries
    'device_type': 5,           # 5 device types
    'hour_of_day': 24,          # 24 hours
    'day_of_week': 7,           # 7 days
}

total_dims = sum(feature_groups.values())
nonzero_per_sample = len(feature_groups)  # One active value per group

print("SPARSE FEATURE REALITY IN AD SYSTEMS")
print("=" * 55)
print(f"\nTotal feature dimensions (one-hot): {total_dims:,}")
print(f"Non-zero features per sample:        {nonzero_per_sample}")
print(f"Sparsity:                            {1 - nonzero_per_sample/total_dims:.8f}")
print(f"\nThat means {100 * (1 - nonzero_per_sample/total_dims):.6f}% of each")
print(f"feature vector is ZEROS!")
print(f"\nIf you stored this densely:")
print(f"  Memory per sample (float32): {total_dims * 4 / 1024 / 1024:.1f} MB")
print(f"  For 1M training samples:     {total_dims * 4 * 1e6 / 1024**4:.1f} TB")
print(f"\nIf you use sparse storage:")
print(f"  Memory per sample: {nonzero_per_sample * 8 / 1024:.2f} KB")
print(f"  That is {total_dims * 4 / (nonzero_per_sample * 8):,.0f}x savings!")

In [ ]:
# Visualize the sparsity pattern
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: What a sparse feature vector looks like (zoomed in)
ax = axes[0]
n_show = 200
sparse_vec = np.zeros(n_show)
# Place a few ones at random positions (like one-hot within groups)
active_positions = [5, 42, 89, 123, 178]
sparse_vec[active_positions] = 1

ax.bar(range(n_show), sparse_vec, width=1, color='#FF6B6B', alpha=0.8)
ax.set_xlabel('Feature Index', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('Sparse Feature Vector (first 200 dims)\n'
             f'Only {len(active_positions)} non-zero out of {n_show}', fontsize=12)
ax.set_ylim(-0.1, 1.5)

# Right: Feature group sizes (pie chart)
ax = axes[1]
top_groups = sorted(feature_groups.items(), key=lambda x: -x[1])[:6]
other_size = sum(v for k, v in feature_groups.items() if (k, v) not in top_groups)
labels = [f'{k}\n({v:,})' for k, v in top_groups] + [f'others\n({other_size:,})']
sizes = [v for _, v in top_groups] + [other_size]
colors = plt.cm.Set3(np.linspace(0, 1, len(sizes)))
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90,
       textprops={'fontsize': 8})
ax.set_title('Feature Dimension Breakdown\n'
             f'Total: {total_dims:,} dimensions', fontsize=12)

plt.tight_layout()
plt.show()

print("Key insight: user_id and ad_id dominate the feature space.")
print("This is why EMBEDDINGS are critical -- we cannot one-hot encode")
print("a billion user IDs in practice.")

## 2. Feature Encoding Techniques

### One-Hot Encoding (Low Cardinality)

For features with a small number of categories (< 1000), one-hot encoding works fine.

```
ad_category = "travel"

One-hot: [0, 0, 0, ..., 1, ..., 0, 0]  # 1 at the "travel" position
             sports  tech       travel     food
```

### Embedding Layers (High Cardinality)

For features with millions of categories (user IDs, ad IDs), we use **embedding layers** that learn a dense vector for each category.

```
user_id = 38291047

Embedding: [0.23, -0.45, 0.12, ..., 0.67]  # 64-dimensional dense vector
```

Think of it like this: instead of saying "this user is user #38291047" (which tells the model nothing), we say "this user is at position (0.23, -0.45, 0.12, ...) in user-taste space" -- which captures their preferences.

In [ ]:
import torch
import torch.nn as nn

# === Embedding Layers in PyTorch ===

class AdFeatureEncoder(nn.Module):
    """
    Encode categorical features using embedding layers.
    
    12-year-old version:
    Each category (like 'user_id=42') gets its own secret code
    (a list of numbers). The model LEARNS these codes during training
    so that similar users get similar codes.
    
    Staff-level detail:
    - Embedding dimensions follow a sqrt rule: dim ~ sqrt(cardinality)
    - Embeddings are shared between the FM and DNN components
    - Regularization (L2) on embeddings prevents overfitting
    - Unknown/new IDs map to a special <UNK> embedding
    """
    def __init__(self):
        super().__init__()
        
        # Each categorical feature gets its own embedding table
        # Rule of thumb: embedding_dim ~ min(sqrt(cardinality), 64)
        self.embeddings = nn.ModuleDict({
            'user_id':        nn.Embedding(100_000, 64),    # 100K users -> 64-dim
            'ad_id':          nn.Embedding(50_000, 32),     # 50K ads -> 32-dim
            'advertiser_id':  nn.Embedding(10_000, 16),     # 10K advertisers -> 16-dim
            'campaign_id':    nn.Embedding(20_000, 16),     # 20K campaigns -> 16-dim
            'ad_category':    nn.Embedding(100, 8),         # 100 categories -> 8-dim
            'ad_subcategory': nn.Embedding(1_000, 12),      # 1K subcategories -> 12-dim
            'device_type':    nn.Embedding(5, 4),           # 5 devices -> 4-dim
        })
        
        # Dense features (already numerical) -- just need normalization
        self.dense_bn = nn.BatchNorm1d(5)  # 5 dense features
    
    def forward(self, categorical_features, dense_features):
        """
        Args:
            categorical_features: dict of {feature_name: tensor of indices}
            dense_features: tensor of [batch_size, 5]
        Returns:
            Concatenated embedding vector
        """
        # Look up embeddings for each categorical feature
        embedded = []
        for name, emb_layer in self.embeddings.items():
            if name in categorical_features:
                embedded.append(emb_layer(categorical_features[name]))
        
        # Normalize dense features
        dense_normalized = self.dense_bn(dense_features)
        
        # Concatenate everything
        all_features = embedded + [dense_normalized]
        return torch.cat(all_features, dim=1)


# Demo
encoder = AdFeatureEncoder()

# Simulate a batch of 4 impressions
batch_size = 4
cat_features = {
    'user_id': torch.randint(0, 100_000, (batch_size,)),
    'ad_id': torch.randint(0, 50_000, (batch_size,)),
    'advertiser_id': torch.randint(0, 10_000, (batch_size,)),
    'campaign_id': torch.randint(0, 20_000, (batch_size,)),
    'ad_category': torch.randint(0, 100, (batch_size,)),
    'ad_subcategory': torch.randint(0, 1_000, (batch_size,)),
    'device_type': torch.randint(0, 5, (batch_size,)),
}
dense_features = torch.randn(batch_size, 5)  # CTR, impressions, etc.

output = encoder(cat_features, dense_features)

print("EMBEDDING ENCODER OUTPUT")
print("=" * 50)
print(f"Input: {len(cat_features)} categorical + 5 dense features")
print(f"Output shape: {output.shape}")
print(f"\nEmbedding dimensions per feature:")
total_emb_dim = 0
for name, emb in encoder.embeddings.items():
    print(f"  {name:<20s}: {emb.num_embeddings:>10,} categories -> {emb.embedding_dim}-dim")
    total_emb_dim += emb.embedding_dim
print(f"  {'dense_features':<20s}: {'5':>10s} values  -> 5-dim")
print(f"  {'TOTAL':<20s}: {total_emb_dim + 5}-dim output vector")
print(f"\nCompare to one-hot: {sum(e.num_embeddings for e in encoder.embeddings.values()):,} dims")
print(f"Embedding reduces dimensions by {sum(e.num_embeddings for e in encoder.embeddings.values()) // (total_emb_dim + 5):,}x!")

## 3. Feature Hashing for High Cardinality

Sometimes you have features with UNBOUNDED cardinality -- like search queries, URLs, or combinations of features. You cannot pre-allocate an embedding table of unknown size.

**Feature hashing** (the "hashing trick") solves this:
1. Hash the feature value to an integer
2. Take modulo with a fixed table size
3. Use that as the embedding index

```
"user_42_clicked_ad_travel" -> hash() -> 7382918 -> 7382918 % 1000000 -> 382918
```

**Pros:** Fixed memory, handles any cardinality, no vocabulary needed
**Cons:** Hash collisions (two features map to same index)

In [ ]:
import hashlib

class FeatureHasher:
    """
    Feature hashing for high-cardinality categorical features.
    
    12-year-old version:
    Imagine you have a wall of 1 million lockers. When a new student
    arrives, instead of assigning them the next available locker,
    you use their name to pick a locker number. "Alice" always maps
    to locker 382918. Sometimes two students get the same locker
    (collision), but with 1M lockers, it rarely happens.
    """
    
    def __init__(self, num_buckets=1_000_000):
        self.num_buckets = num_buckets
    
    def hash_feature(self, feature_value):
        """Hash a feature value to a bucket index."""
        h = hashlib.md5(str(feature_value).encode()).hexdigest()
        return int(h, 16) % self.num_buckets
    
    def hash_cross(self, *features):
        """Hash a combination of features (feature cross)."""
        combined = "_X_".join(str(f) for f in features)
        return self.hash_feature(combined)


hasher = FeatureHasher(num_buckets=1_000_000)

# Demo: Hash individual features
features_to_hash = [
    'user_42', 'user_999999', 'ad_travel_hotel', 
    'campaign_summer2024', 'some_totally_new_feature_never_seen_before'
]

print("FEATURE HASHING DEMO")
print("=" * 60)
for f in features_to_hash:
    bucket = hasher.hash_feature(f)
    print(f"  '{f}' -> bucket {bucket:,}")

# Demo: Hash feature crosses
print(f"\nFEATURE CROSS HASHING")
print("=" * 60)
crosses = [
    ('user_42', 'ad_category_travel'),
    ('user_42', 'ad_category_insurance'),
    ('country_US', 'hour_14', 'device_mobile'),
]
for cross in crosses:
    bucket = hasher.hash_cross(*cross)
    print(f"  {' x '.join(cross)} -> bucket {bucket:,}")

# Collision analysis
print(f"\nCOLLISION ANALYSIS")
print("=" * 60)
n_features = 10_000_000  # 10M unique features
n_buckets = 1_000_000    # 1M buckets
# Expected collision rate approximation: 1 - e^(-n/m)
import math
collision_rate = 1 - math.exp(-n_features / n_buckets)
print(f"  Unique features:  {n_features:,}")
print(f"  Hash buckets:     {n_buckets:,}")
print(f"  Expected collisions: ~{collision_rate:.1%} of buckets will have >1 feature")
print(f"\n  Rule of thumb: Use 10x-100x more buckets than expected features")

## 4. Feature Crosses (The Secret Sauce)

Feature crosses combine two or more features to create a new feature that captures their **interaction**.

### Why Crosses Matter

Think of it like ingredients in cooking:
- **Salt** alone = salty
- **Chocolate** alone = sweet
- **Salt + Chocolate** together = AMAZING salted chocolate

The combination creates something neither ingredient can express alone.

### Common Crosses in Ad Systems

| Cross | Why It Helps |
|-------|-------------|
| `user_age x ad_category` | Young users like gaming ads, older users like insurance |
| `user_country x ad_language` | US users prefer English ads |
| `hour_of_day x device_type` | Mobile usage peaks during commute hours |
| `user_gender x ad_subcategory` | Gendered product preferences |
| `user_clicked_categories x ad_category` | User interest alignment |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Feature Crossing Demo ===
# Show how crosses capture interactions that individual features miss

np.random.seed(42)

# Simulate users and their click behavior
n_samples = 10000

# Features
age_group = np.random.choice(['18-24', '25-34', '35-44', '45+'], n_samples)
ad_category = np.random.choice(['gaming', 'insurance', 'travel', 'fashion'], n_samples)
hour = np.random.randint(0, 24, n_samples)
device = np.random.choice(['mobile', 'desktop'], n_samples)

# True CTR depends on INTERACTIONS between features
ctr_map = {
    ('18-24', 'gaming'): 0.12,   ('18-24', 'insurance'): 0.01,
    ('18-24', 'travel'): 0.04,   ('18-24', 'fashion'): 0.08,
    ('25-34', 'gaming'): 0.06,   ('25-34', 'insurance'): 0.05,
    ('25-34', 'travel'): 0.08,   ('25-34', 'fashion'): 0.06,
    ('35-44', 'gaming'): 0.02,   ('35-44', 'insurance'): 0.09,
    ('35-44', 'travel'): 0.06,   ('35-44', 'fashion'): 0.03,
    ('45+', 'gaming'): 0.01,     ('45+', 'insurance'): 0.08,
    ('45+', 'travel'): 0.10,     ('45+', 'fashion'): 0.02,
}

# Generate clicks
true_ctr = np.array([ctr_map[(a, c)] for a, c in zip(age_group, ad_category)])
clicks = (np.random.random(n_samples) < true_ctr).astype(int)

df = pd.DataFrame({
    'age_group': age_group, 'ad_category': ad_category,
    'hour': hour, 'device': device, 'clicked': clicks
})

# Show the interaction effect
pivot = df.groupby(['age_group', 'ad_category'])['clicked'].mean().unstack()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Marginal effects (individual features)
ax = axes[0]
age_effect = df.groupby('age_group')['clicked'].mean()
cat_effect = df.groupby('ad_category')['clicked'].mean()
x = np.arange(4)
ax.bar(x - 0.2, age_effect.values, 0.35, label='By Age Group', color='#FF6B6B', alpha=0.8)
ax.bar(x + 0.2, cat_effect.values, 0.35, label='By Ad Category', color='#4ECDC4', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['G1', 'G2', 'G3', 'G4'])
ax.set_ylabel('Average CTR', fontsize=12)
ax.set_title('Individual Feature Effects\n(Marginal -- misleading!)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Right: Interaction effects (crossed features)
ax = axes[1]
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=10)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=10)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f'{val:.1%}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if val > 0.06 else 'black')
ax.set_xlabel('Ad Category', fontsize=12)
ax.set_ylabel('Age Group', fontsize=12)
ax.set_title('CROSSED Feature Effects\n(Age x Category -- the real story!)', fontsize=12)
plt.colorbar(im, label='CTR')

plt.tight_layout()
plt.show()

print("Left plot: individual features look similar across groups.")
print("Right plot: the COMBINATION reveals dramatic differences!")
print("\n18-24 + gaming = 12% CTR, but 18-24 + insurance = 1% CTR.")
print("This is why feature crosses are essential for ad click prediction.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import train_test_split

# === Quantify the impact of feature crosses ===

# Model 1: Without feature crosses
X_no_cross = pd.get_dummies(df[['age_group', 'ad_category', 'device']])
X_no_cross['hour'] = df['hour']
y = df['clicked']

# Model 2: With feature crosses
df['age_x_category'] = df['age_group'] + '_' + df['ad_category']
df['age_x_device'] = df['age_group'] + '_' + df['device']
df['category_x_device'] = df['ad_category'] + '_' + df['device']

X_with_cross = pd.get_dummies(df[['age_group', 'ad_category', 'device',
                                   'age_x_category', 'age_x_device', 'category_x_device']])
X_with_cross['hour'] = df['hour']

# Train and compare
results = {}
for name, X_data in [('Without Crosses', X_no_cross), ('With Crosses', X_with_cross)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X_data, y, test_size=0.2, random_state=42)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_tr, y_tr)
    y_pred = model.predict_proba(X_te)[:, 1]
    
    auc = roc_auc_score(y_te, y_pred)
    ce = log_loss(y_te, y_pred)
    results[name] = {'AUC': auc, 'LogLoss': ce, 'n_features': X_data.shape[1]}

print("IMPACT OF FEATURE CROSSES ON LOGISTIC REGRESSION")
print("=" * 55)
for name, r in results.items():
    print(f"\n{name}:")
    print(f"  Features:  {r['n_features']}")
    print(f"  AUC-ROC:   {r['AUC']:.4f}")
    print(f"  Log Loss:  {r['LogLoss']:.4f}")

auc_lift = results['With Crosses']['AUC'] - results['Without Crosses']['AUC']
print(f"\nAUC improvement from crosses: +{auc_lift:.4f}")
print(f"\nIn real ad systems, this kind of improvement = millions of dollars.")

## 5. Real-Time vs Batch Features

### The Two Feature Pipelines

Think of it like a restaurant kitchen:
- **Batch features** = ingredients you prep the night before (sauces, marinades, chopped vegetables)
- **Real-time features** = things you cook to order (grilled steak, fresh salad)

| Type | Examples | Update Frequency | Storage | Latency Budget |
|------|----------|-----------------|---------|----------------|
| **Batch** | Ad image embedding, category, user demographics | Every few hours/days | Feature Store | N/A (pre-computed) |
| **Real-time** | Ad impressions in last hour, user's last 5 clicks, session length | Every request | Computed on-the-fly | < 10ms |

In [ ]:
import time
from collections import defaultdict

# === Simulating Batch vs Real-Time Feature Computation ===

class FeatureStore:
    """
    Simulates a feature store that serves batch-computed features.
    
    12-year-old version:
    Think of this as a giant filing cabinet. Every night,
    a robot computes features for every user and ad and
    files them away. When we need them, we just look them up
    -- super fast!
    """
    def __init__(self):
        self.user_features = {}  # Pre-computed user features
        self.ad_features = {}    # Pre-computed ad features
    
    def precompute_user_features(self, user_id):
        """Batch-compute user features (runs overnight)."""
        return {
            'user_id': user_id,
            'age_bucket': np.random.choice(['18-24', '25-34', '35-44', '45+']),
            'gender': np.random.choice(['M', 'F']),
            'country': np.random.choice(['US', 'UK', 'CA', 'DE']),
            'historical_ctr': round(np.random.beta(2, 50), 4),
            'total_clicks_30d': np.random.randint(0, 100),
            'preferred_categories': np.random.choice(
                ['travel', 'tech', 'gaming', 'fashion'], size=3).tolist(),
        }
    
    def precompute_ad_features(self, ad_id):
        """Batch-compute ad features (runs when ad is created + periodic refresh)."""
        return {
            'ad_id': ad_id,
            'category': np.random.choice(['travel', 'tech', 'gaming', 'fashion']),
            'advertiser_quality_score': round(np.random.uniform(0.3, 1.0), 2),
            'image_embedding': np.random.randn(64).tolist(),  # SimCLR output
            'historical_ctr': round(np.random.beta(2, 50), 4),
            'total_impressions': np.random.randint(100, 100_000),
        }
    
    def populate(self, n_users=100, n_ads=50):
        """Simulate overnight batch job."""
        for uid in range(n_users):
            self.user_features[uid] = self.precompute_user_features(uid)
        for aid in range(n_ads):
            self.ad_features[aid] = self.precompute_ad_features(aid)
    
    def get_user(self, user_id):
        return self.user_features.get(user_id, None)
    
    def get_ad(self, ad_id):
        return self.ad_features.get(ad_id, None)


class RealTimeFeatureComputer:
    """
    Computes features that change too fast for batch processing.
    
    12-year-old version:
    Some things change every second: how many times this ad
    was shown in the last hour, what the user clicked 5 minutes
    ago, whether they are on Wi-Fi or cellular right now.
    These must be computed fresh for every request.
    """
    def __init__(self):
        self.recent_clicks = defaultdict(list)  # user -> recent ad clicks
        self.session_data = {}                  # user -> current session info
    
    def compute_realtime_features(self, user_id, ad_id, context):
        """Compute real-time features (<10ms budget!)."""
        return {
            # User session features
            'session_ad_views': np.random.randint(0, 20),
            'session_ad_clicks': np.random.randint(0, 3),
            'seconds_since_last_click': np.random.randint(10, 3600),
            
            # Ad freshness features
            'ad_impressions_last_hour': np.random.randint(0, 1000),
            'ad_clicks_last_hour': np.random.randint(0, 50),
            'ad_ctr_last_hour': round(np.random.beta(2, 50), 4),
            
            # Context features
            'hour_of_day': context.get('hour', 12),
            'day_of_week': context.get('day', 3),
            'is_weekend': context.get('day', 3) >= 5,
            'device': context.get('device', 'mobile'),
            
            # User-ad real-time features
            'times_shown_to_user': np.random.randint(0, 5),
        }


# Run the simulation
feature_store = FeatureStore()
rt_computer = RealTimeFeatureComputer()

# Simulate batch pre-computation
t0 = time.time()
feature_store.populate(n_users=100, n_ads=50)
batch_time = time.time() - t0

# Simulate real-time feature computation
t0 = time.time()
for _ in range(100):  # 100 requests
    batch_user = feature_store.get_user(42)
    batch_ad = feature_store.get_ad(7)
    rt_features = rt_computer.compute_realtime_features(
        42, 7, {'hour': 14, 'day': 3, 'device': 'mobile'})
rt_time = time.time() - t0

print("FEATURE COMPUTATION PIPELINE")
print("=" * 60)
print(f"\nBatch Pre-computation (100 users, 50 ads): {batch_time*1000:.1f}ms")
print(f"Real-time computation (100 requests):       {rt_time*1000:.1f}ms")
print(f"Real-time per request:                      {rt_time*1000/100:.3f}ms")

print(f"\n--- Sample Combined Feature Vector ---")
print(f"\nBatch (user):")
for k, v in batch_user.items():
    if k != 'preferred_categories':
        print(f"  {k}: {v}")
print(f"\nBatch (ad):")
for k, v in batch_ad.items():
    if k != 'image_embedding':
        print(f"  {k}: {v}")
print(f"\nReal-time:")
for k, v in rt_features.items():
    print(f"  {k}: {v}")

## 6. Negative Downsampling and Calibration

### The Problem: Extreme Class Imbalance

In ad click prediction, the typical CTR is 1-5%. That means for every 100 impressions, only 1-5 are clicks. This creates a **huge class imbalance** -- 95-99% of your training data is negative.

### Negative Downsampling

Instead of training on ALL negatives, randomly keep only a fraction of them.

```
Original: 1M positives + 99M negatives (1% CTR)
After 10x downsampling: 1M positives + 9.9M negatives (10% effective CTR)
```

**Benefits:**
- Faster training (10x fewer samples)
- Better gradient signal (model sees more balanced data)

**But wait!** Now our predicted probabilities are wrong -- the model thinks CTR is 10% when it is really 1%. We need **calibration** to fix this.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Negative Downsampling + Calibration ===

def downsample_negatives(y, negative_rate=0.1):
    """
    Downsample negative examples to balance the dataset.
    
    12-year-old version:
    If you have 100 photos and only 2 are of cats (positives),
    it is hard to learn what cats look like. So you throw away
    most of the non-cat photos to make it more balanced.
    
    Args:
        y: array of labels (0 or 1)
        negative_rate: fraction of negatives to keep
    Returns:
        mask: boolean array of which samples to keep
    """
    mask = np.ones(len(y), dtype=bool)
    neg_indices = np.where(y == 0)[0]
    n_keep = int(len(neg_indices) * negative_rate)
    drop_indices = np.random.choice(neg_indices, len(neg_indices) - n_keep, replace=False)
    mask[drop_indices] = False
    return mask


def calibrate_predictions(p_sampled, negative_rate):
    """
    Calibrate predictions after negative downsampling.
    
    Formula: p_calibrated = p_sampled / (p_sampled + (1 - p_sampled) / negative_rate)
    
    This reverses the effect of downsampling so the predicted
    probabilities match the TRUE click-through rate.
    
    Why calibration matters in ad systems:
    - Revenue = sum(pCTR * bid) for shown ads
    - If pCTR is 10x too high, the auction is broken
    - Advertisers are charged based on pCTR, so it MUST be accurate
    """
    return p_sampled / (p_sampled + (1 - p_sampled) / negative_rate)


# Demo: Create imbalanced dataset
np.random.seed(42)
n = 100_000
true_ctr = 0.02  # 2% CTR (typical for social media ads)
y = (np.random.random(n) < true_ctr).astype(int)

print("NEGATIVE DOWNSAMPLING AND CALIBRATION")
print("=" * 55)
print(f"\nOriginal dataset:")
print(f"  Total samples:    {n:,}")
print(f"  Positives:        {y.sum():,} ({y.mean():.2%})")
print(f"  Negatives:        {(y == 0).sum():,} ({1 - y.mean():.2%})")

# Apply downsampling
neg_rate = 0.1  # Keep only 10% of negatives
mask = downsample_negatives(y, negative_rate=neg_rate)
y_sampled = y[mask]

print(f"\nAfter downsampling (keep {neg_rate:.0%} of negatives):")
print(f"  Total samples:    {len(y_sampled):,}")
print(f"  Positives:        {y_sampled.sum():,} ({y_sampled.mean():.2%})")
print(f"  Negatives:        {(y_sampled == 0).sum():,} ({1 - y_sampled.mean():.2%})")

# Simulate model predictions on the sampled data
# The model will predict ~17% average CTR (inflated!)
sampled_ctr = y_sampled.mean()
p_sampled = np.clip(np.random.normal(sampled_ctr, 0.05, 1000), 0.01, 0.99)

# Calibrate back to true probabilities
p_calibrated = calibrate_predictions(p_sampled, neg_rate)

print(f"\nCalibration effect:")
print(f"  Model avg prediction (before):  {p_sampled.mean():.4f}")
print(f"  Model avg prediction (after):   {p_calibrated.mean():.4f}")
print(f"  True CTR:                       {true_ctr:.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(p_sampled, bins=50, alpha=0.7, color='#FF6B6B', label='Before calibration')
axes[0].hist(p_calibrated, bins=50, alpha=0.7, color='#4ECDC4', label='After calibration')
axes[0].axvline(true_ctr, color='black', linestyle='--', linewidth=2, label=f'True CTR ({true_ctr:.2%})')
axes[0].set_xlabel('Predicted P(click)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Prediction Distribution Before/After Calibration', fontsize=12)
axes[0].legend(fontsize=10)

# Calibration curve
rates = np.linspace(0.01, 1.0, 100)
p_example = 0.15  # Model predicts 15%
calibrated = calibrate_predictions(p_example, rates)
axes[1].plot(rates, calibrated, 'b-', linewidth=2)
axes[1].set_xlabel('Negative Sampling Rate', fontsize=12)
axes[1].set_ylabel('Calibrated Probability', fontsize=12)
axes[1].set_title(f'Calibration Curve\n(Model prediction = {p_example:.0%})', fontsize=12)
axes[1].axhline(p_example, color='red', linestyle='--', alpha=0.5, label='Uncalibrated')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

Which features actually MATTER? In ad systems, understanding feature importance is critical for:
- **Debugging:** If the model suddenly performs poorly, which feature broke?
- **Latency:** Can we drop expensive real-time features without losing accuracy?
- **Interview:** Being able to rank features by importance shows system-level thinking.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np
import matplotlib.pyplot as plt

# === Feature Importance with a Full Feature Set ===

np.random.seed(42)
n = 20000

# Generate realistic features
feature_names = [
    # User features
    'user_historical_ctr', 'user_total_clicks_30d', 'user_session_clicks',
    'user_age_bucket', 'user_is_mobile',
    # Ad features
    'ad_historical_ctr', 'ad_total_impressions', 'ad_category_match',
    'ad_quality_score', 'ad_freshness_days',
    # Context features
    'hour_of_day', 'is_weekend', 'position_on_page',
    # Cross features
    'user_age_x_ad_category', 'user_ctr_x_ad_ctr',
]

X = np.column_stack([
    np.random.beta(2, 50, n),              # user_historical_ctr
    np.random.poisson(10, n),               # user_total_clicks_30d
    np.random.poisson(1, n),                # user_session_clicks
    np.random.randint(0, 4, n),             # user_age_bucket
    np.random.binomial(1, 0.7, n),          # user_is_mobile
    np.random.beta(2, 50, n),              # ad_historical_ctr
    np.random.exponential(10000, n),         # ad_total_impressions
    np.random.binomial(1, 0.3, n),          # ad_category_match
    np.random.uniform(0.3, 1.0, n),         # ad_quality_score
    np.random.exponential(30, n),            # ad_freshness_days
    np.random.randint(0, 24, n),            # hour_of_day
    np.random.binomial(1, 0.3, n),          # is_weekend
    np.random.randint(1, 6, n),             # position_on_page
    np.random.randn(n),                     # user_age_x_ad_category
    np.random.randn(n),                     # user_ctr_x_ad_ctr
])

# Generate labels with known feature importance
logits = (
    -3.0
    + 8.0 * X[:, 0]            # user_historical_ctr (STRONG)
    + 0.05 * X[:, 1]           # user_total_clicks_30d
    + 0.2 * X[:, 2]            # user_session_clicks
    + 5.0 * X[:, 5]            # ad_historical_ctr (STRONG)
    + 0.8 * X[:, 7]            # ad_category_match (MODERATE)
    + 0.5 * X[:, 8]            # ad_quality_score
    - 0.3 * X[:, 12]           # position_on_page (MODERATE, negative)
    + 0.6 * X[:, 13]           # user_age_x_ad_category cross
    + 0.4 * X[:, 14]           # user_ctr_x_ad_ctr cross
    + np.random.normal(0, 0.3, n)
)
y = (np.random.random(n) < 1 / (1 + np.exp(-logits))).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train GBDT
model = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)

print(f"GBDT Model AUC: {auc:.4f}")

# Method 1: Built-in feature importance (impurity-based)
importances = model.feature_importances_

# Method 2: Permutation importance (more reliable)
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)

# Visualize both
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sort by importance
sorted_idx_builtin = np.argsort(importances)
sorted_idx_perm = np.argsort(perm_importance.importances_mean)

colors_builtin = ['#FF6B6B' if importances[i] > 0.1 else '#4ECDC4' for i in sorted_idx_builtin]
axes[0].barh(range(len(feature_names)), importances[sorted_idx_builtin], color=colors_builtin)
axes[0].set_yticks(range(len(feature_names)))
axes[0].set_yticklabels([feature_names[i] for i in sorted_idx_builtin], fontsize=9)
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('Built-in Feature Importance\n(Impurity-based)', fontsize=12)

colors_perm = ['#FF6B6B' if perm_importance.importances_mean[i] > 0.01 else '#4ECDC4' for i in sorted_idx_perm]
axes[1].barh(range(len(feature_names)), perm_importance.importances_mean[sorted_idx_perm], color=colors_perm)
axes[1].set_yticks(range(len(feature_names)))
axes[1].set_yticklabels([feature_names[i] for i in sorted_idx_perm], fontsize=9)
axes[1].set_xlabel('Mean AUC Decrease', fontsize=12)
axes[1].set_title('Permutation Importance\n(More reliable)', fontsize=12)

plt.suptitle('Feature Importance Analysis for Ad Click Prediction', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nTop features (as expected):")
print("  1. user_historical_ctr -- past behavior is the best predictor")
print("  2. ad_historical_ctr -- good ads keep performing well")
print("  3. ad_category_match -- relevance to user interests")
print("  4. Cross features -- capture non-linear interactions")

## Interview Cheat Sheet: Feature Engineering

### Key Phrases to Drop

- "The feature space is extremely **sparse** with millions of categorical features, so we use **embedding layers** to learn dense representations."
- "Feature crosses like `user_age x ad_category` capture **interaction effects** that individual features cannot express."
- "We use **feature hashing** for unbounded cardinality features to maintain fixed memory."
- "Batch features are pre-computed in a **feature store**, while real-time features (session stats, recency) are computed at query time within a **10ms budget**."
- "After **negative downsampling**, we apply **calibration** (Platt scaling or the analytical formula) because predicted probabilities feed directly into the ad **auction**."

### Common Pitfalls

| Pitfall | Why It Is Wrong | What to Say Instead |
|---------|-----------------|--------------------|
| One-hot encode user IDs | 1B dimensions would explode memory | Use embedding layers |
| Only use individual features | Misses interactions | Use feature crosses or FM/DCN |
| Skip calibration | Auction breaks with uncalibrated pCTR | Always calibrate after downsampling |
| All features real-time | Too slow (>100ms) | Split into batch vs real-time |
| Ignore position bias | Position 1 always gets more clicks | Include position as a feature or use IPW |

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)